# VENEER — how much of an LLM judge's verdict is just formatting?

**Same facts. Different clothes. Watch the judge change its mind.**

Every answer in this dataset is built from *one fixed list of atomic claims*. Nine renderers turn that same list into plain prose, bullets, markdown headings, a table, bolded terms, emoji, and so on. **No renderer adds, removes or alters a claim** — the harness machine-checks that.

So when a judge prefers one rendering over another, it is not preferring better content. There is no better content. It is preferring *clothes*.

This notebook walks the result in five steps.

In [ ]:
import json, os, glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Kaggle's real mount path is nested (/kaggle/input/datasets/<owner>/<slug>/),
# so walk it rather than assuming any one layout. Falls back to a local checkout.
BASE = None
for root, _, files in os.walk('/kaggle/input'):
    if 'judgments.csv' in files:
        BASE = root; break
if BASE is None:
    BASE = next(p for p in ['release/', '../release/', './']
                if os.path.exists(os.path.join(p, 'judgments.csv')))
print('data dir:', BASE)
L = lambda f: pd.read_csv(os.path.join(BASE, f))

items, rend = L('items.csv'), L('renderings.csv')
jud, summ, lb = L('judgments.csv'), L('summary.csv'), L('leaderboard.csv')
print(f'{len(items)} items · {len(rend)} renderings · {len(jud):,} judgments · {jud.judge.nunique()} judges')

## 1. The setup — one claim list, nine outfits

Look at the same six facts wearing different clothes. Nothing is added. Nothing is removed.

In [ ]:
ex = items.iloc[0]
print('QUESTION:', ex.question, '\n')
print('THE SUBSTANCE (identical in every rendering below):')
for c in json.loads(ex.claims):
    print('  •', c)

for r in ['plain', 'bullets', 'markdown_max']:
    txt = rend[(rend.id == ex.id) & (rend.rendering == r)].answer.iloc[0]
    print('\n' + '=' * 70 + f'\n{r.upper()}\n' + '=' * 70)
    print(txt[:700])

### Verify it yourself: the substance really is identical

Compare the multiset of content words against the source claims. Only content-free scaffolding (generic headings, table labels, neutral padding) is allowed to differ.

In [ ]:
import re, collections
# Exactly the content-free scaffolding vocabulary the renderers may introduce
# (generic headings, table labels, neutral padding). Derived from src/render.py.
SCAFFOLD = set('a additional another as broadly case circumstances considerations context cover detail details essentials established further general generally here holds in information is it keeping key main matters mind more most noting of other overview point points practical put question relevant rule simply situation speaking summary taken terms that the these this to together usual way well worth'.split())
SCAFFOLD |= {str(i) for i in range(1, 60)}
sig = lambda t: collections.Counter(re.findall(r'[a-z0-9]+', t.lower()))

leaks = 0
for _, it in items.iterrows():
    base = sig(' '.join(json.loads(it.claims)))
    for _, r in rend[rend.id == it.id].iterrows():
        extra = {w: n for w, n in (sig(r.answer) - base).items() if w not in SCAFFOLD}
        if extra or (base - sig(r.answer)):
            leaks += 1
print(f'renderings checked: {len(rend)}   substance violations: {leaks}')

## 2. The headline — formatting moves the verdict

Win rate is the share of non-tie decisions in which the judge picked the formatted version over identical-content plain prose. **50% is indifference.**

The grey line is the *noise floor*: plain prose judged against a byte-identical copy of itself. Any real effect has to clear it.

In [ ]:
COL = {'control': '#8a8a8a', 'structure': '#3b7dd8', 'emphasis': '#d9534f', 'length': '#e0a800'}
t = summ.sort_values('win_rate')
floor = float(summ.loc[summ.rendering == 'plain_vs_plain', 'win_rate'].iloc[0])

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.barh(t.rendering, t.win_rate, color=[COL[a] for a in t.axis], height=.68)
ax.errorbar(t.win_rate, range(len(t)), xerr=[t.win_rate - t.ci_lo, t.ci_hi - t.win_rate],
            fmt='none', ecolor='#222', elinewidth=1.1, capsize=3)
ax.axvline(.5, color='#222', lw=1, ls='--')
ax.axvline(floor, color='#8a8a8a', lw=1.5, ls=':')
ax.set_xlim(0, 1); ax.set_xlabel('win rate vs. identical-content plain prose')
ax.set_title('Presentation alone moves an LLM judge', fontsize=13, weight='bold')
for s in ('top', 'right'): ax.spines[s].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=COL[a]) for a in COL], list(COL),
          title='axis', frameon=False, loc='lower right')
plt.tight_layout(); plt.show()

summ[['rendering','axis','n','win_rate','ci_lo','ci_hi','tie_rate','deviation_pp','differs_from_50']].round(3)

## 3. The leaderboard — which judges are format-blind?

**VENEER score** = mean absolute deviation from a 50% win rate across all format conditions, in percentage points. It is how much of a verdict presentation alone can buy.

**0 = perfectly format-blind. Lower is better — with one big caveat.**

A judge whose verdict is decided by *where* an answer sits produces a win rate pulled toward 50% in *every* condition, which looks identical to genuine format-blindness. So the leaderboard always carries `first_pick_rate` and `position_gap`. **The lowest VENEER score in this table is a position-confounded judge, not the best one** — cell 5 below shows the split that proves it.

In [ ]:
display(lb)
fig, ax = plt.subplots(figsize=(7, 3.2))
b = ax.barh(lb.judge, lb.VENEER_score, color='#3b7dd8', height=.6)
ax.bar_label(b, fmt='%.1f', padding=4, fontsize=10)
ax.set_xlabel('VENEER score (pp of verdict buyable with formatting) — lower is better')
ax.set_title('Judge format-robustness', fontsize=12, weight='bold')
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

### The check that decides it

If a judge's variant-wins land overwhelmingly in one position bucket and collapse in the other, its flat VENEER score is measuring seating, not format-robustness.


In [ ]:
# Does the variant win because of its FORMAT, or because of WHERE it sat?
d = jud[jud.winner != 'tie'].copy()
d['j'] = d.judge.str.replace('claude-','').str.replace('-4-5-20251001','')
v = d[d.rendering != 'plain_vs_plain']
for j in sorted(v.j.unique()):
    s = v[v.j == j]
    second = (s[s.plain_first].winner == 'variant').mean()
    first  = (s[~s.plain_first].winner == 'variant').mean()
    flag = '  <-- POSITION-DOMINATED' if abs(second-first) > .5 else ''
    print(f'{j:10s} variant wins {second:.3f} when shown 2nd | {first:.3f} when shown 1st'
          f' | gap {second-first:+.3f}{flag}')


## 4. Does susceptibility differ by judge and by domain?

In [ ]:
d = jud[jud.winner != 'tie'].assign(w=lambda x: x.winner == 'variant')
piv = d.pivot_table(index='rendering', columns='judge', values='w')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(piv.values, cmap='RdBu_r', vmin=.2, vmax=.8, aspect='auto')
ax.set_xticks(range(len(piv.columns)), [c.replace('claude-','') for c in piv.columns], rotation=20)
ax.set_yticks(range(len(piv.index)), piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(v-.5) > .18 else '#111')
ax.set_title('Win rate vs plain, by judge', fontsize=12, weight='bold')
fig.colorbar(im, ax=ax, shrink=.8); plt.tight_layout(); plt.show()

print('\nWin rate by domain (all formats pooled):')
print(d[d.rendering != 'plain_vs_plain'].groupby('domain').w.mean().round(3).to_string())

## 5. Sanity check — position bias

Presentation order is randomised per (item, rendering, judge) by a fixed seed. If a judge simply picked whichever answer appeared first, this would sit far from 0.5 and the win rates above would be an artifact.

In [ ]:
pb = jud[jud.winner != 'tie'].assign(first_pick=lambda x: x.raw == 'A').groupby('judge')['first_pick'].mean()
print('P[picks the answer shown first]   (0.50 = unbiased)')
print(pb.round(3).to_string())
print('\nControl condition (plain vs a byte-identical copy of itself):')
print(summ[summ.rendering == 'plain_vs_plain'][['win_rate','ci_lo','ci_hi','tie_rate']].round(3).to_string(index=False))

## So what?

LLM-as-judge is load-bearing in RLHF preference data, eval harnesses, agent self-critique and model-selection decisions.

If a verdict moves this far on presentation alone, then any pipeline that ranks *model outputs* with an LLM judge is partly ranking *house style* — and a model trained on that signal learns to format, not to be right.

**Two things you can do with this dataset today:**
1. Score your own judge (prompt, model, or rubric) and add it to the leaderboard — the harness takes one line to add a judge.
2. Use it as a regression test: a judge prompt that hardens against format bias should push its VENEER score toward 0 without losing accuracy on real quality differences.

Harness, renderers and the substance checker: **github.com/uditjainstjis/veneer-bench**